In [1]:
import math, random, os, sys, time
import numpy as np
import torch
import torch.nn as nn
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [2]:
class RoPE(nn.Module):
    def __init__(self, d_h, T_m = 2048, base=10000):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, d_h, 2).float() / d_h))
        pos = torch.arange(T_m).float()
        freqs = torch.outer(pos, inv_freq)
        cos = freqs.cos()
        sin = freqs.sin()
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)
    def _rotate_half(self, x):
        x_even = x[..., 0::2]
        x_odd = x[..., 1::2]
        return torch.stack((-x_odd, x_even), dim=-1).flatten(-2)
    def forward(self, x):
        # (b, t, n_h, d_h)
        t = x.shape[1]
        cos = self.cos[:t]
        sin = self.sin[:t]
        cos = torch.repeat_interleave(cos, 2, dim=-1)[None, :, None, :]
        sin = torch.repeat_interleave(sin, 2, dim=-1)[None, :, None, :]
        return (x * cos + self._rotate_half(x) * sin)

In [30]:
x = torch.rand(2, 3, 3, 4)
rope = RoPE(d_h = x.shape[-1])
y = rope(x)
# print(x)
# print(y)
b, t, nh, dh = x.shape
for i in range(b*t*nh):
    i1 = i // (t * nh)
    i2 = (i % (t * nh)) // nh
    i3 = (i % (t * nh)) % nh
    x1 = x[i1, i2, i3, :]
    x2 = y[i1, i2, i3, :]
    print(f"{i1, i2, i3}: ", end='')
    print(f"{torch.rad2deg(torch.arccos(torch.dot(x1, x2) / torch.sqrt(torch.dot(x1, x1) * torch.dot(x2, x2)))).item():.3f}")


(0, 0, 0): 0.000
(0, 0, 1): 0.000
(0, 0, 2): 0.000
(0, 1, 0): 48.762
(0, 1, 1): 42.045
(0, 1, 2): 54.133
(0, 2, 0): 76.755
(0, 2, 1): 82.168
(0, 2, 2): 37.947
(1, 0, 0): 0.000
(1, 0, 1): 0.000
(1, 0, 2): 0.000
(1, 1, 0): 42.606
(1, 1, 1): 52.933
(1, 1, 2): 22.116
(1, 2, 0): 110.127
(1, 2, 1): 74.689
(1, 2, 2): 107.271


In [26]:
x1 = np.array([0.5161, 0.3939])
x2 = np.array([0.5161, 0.3939])
print(np.dot(x1, x1))
print(np.dot(x2, x2))
print(np.rad2deg(np.arccos(np.dot(x1, x2) / np.dot(x1, x1))))


0.42151642
0.42151642
0.0


In [39]:
import threading
from transformers import AutoTokenizer, AutoModelForCausalLM, TextIteratorStreamer
from time import time
import psutil

tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2")
prompt = "The next day is bright"
tokens = tokenizer.encode(prompt, return_tensors="pt")
def get_ram_mb():
  """Returns current process RAM usage in MB."""
  return psutil.Process().memory_info().rss / (1024**2)
def stream_output(use_cache):
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    streamer = TextIteratorStreamer(tokenizer, skip_special_tokens=True)
    thread = threading.Thread(
        target=model.generate,
        kwargs={
            "input_ids": tokens,
            "max_new_tokens": 100,
            "use_cache": use_cache,
            "streamer": streamer,
            # Sampling & Quality Controls:
            # "do_sample": True,  # Enables random sampling instead of greedy picking
            # "temperature": 0.7,  # Controls creativity (lower = focused, higher = creative)
            # "top_k": 50,  # Limits options to top 50 most likely tokens
            # "top_p": 0.9,  # Nucleus sampling: cuts off low-probability tail
            "repetition_penalty": 1.2,  # Discourages re-using recent tokens
            "no_repeat_ngram_size": 2,  # Hard blocks repeating 2-word sequences
        },
    )
    ram_start = get_ram_mb()
    start_time = time()
    thread.start()
    for token in streamer:
        print(token, end="", flush=True)
    end_time = time()
    thread.join()
    elapsed = end_time - start_time
    ram_end = get_ram_mb()
    print("\n" + "-" * 45)
    print(f"use_cache: {use_cache}")
    print(f"Time taken: {elapsed:.4f}s")
    print(
        f"System RAM (RSS): {ram_end:.2f} MB (Delta:"
        f" {ram_end - ram_start:+.2f} MB)"
    )
    if torch.cuda.is_available():
        vram_curr = torch.cuda.memory_allocated() / (1024**2)
        vram_peak = torch.cuda.max_memory_allocated() / (1024**2)
        print(f"VRAM Allocated: {vram_curr:.2f} MB | Peak VRAM: {vram_peak:.2f} MB")
    else:
        print("VRAM: CUDA unavailable (running on CPU)")
    print("-" * 45 + "\n")

stream_output(False)
stream_output(True)

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 4313.12it/s]
[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


The next day is bright and sunny, but the sun has already set. The wind blows in from all directions as it passes through our house on a regular basis."
"I'm not sure what to say," I said with an air of disbelief that was almost too much for my voice. "But you're right about this one: It's going down at least three times faster than normal today!" He looked up into his eyes again before he turned back around once more; there were no clouds or rainbows here either

[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



---------------------------------------------
use_cache: False
Time taken: 9.5462s
System RAM (RSS): 903.04 MB (Delta: +462.75 MB)
VRAM Allocated: 0.00 MB | Peak VRAM: 0.00 MB
---------------------------------------------

The next day is bright and sunny, but the sun has already set. The wind blows in from all directions as it passes through our house on a regular basis."
"I'm not sure what to say," I said with an air of disbelief that was almost too much for my voice. "But you're right about this one: It's going down at least three times faster than normal today!" He looked up into his eyes again before he turned back around once more; there were no clouds or rainbows here either
---------------------------------------------
use_cache: True
Time taken: 5.6691s
System RAM (RSS): 913.80 MB (Delta: +10.76 MB)
VRAM Allocated: 0.00 MB | Peak VRAM: 0.00 MB
---------------------------------------------

